# Phase 2D — Official-demo temporal graph dataset 생성

**목적:** official demonstration을 replay해 temporal graph와 holding weak-label artifact를 확인한다.  
**입력:** LIBERO HDF5/BDDL, fixed split manifest, Phase 2A graph spec.  
**이전 phase:** Phase 2A graph contract; Phase 2R은 diagnostic reference만 제공한다.  
**출력:** per-demo shards, merged natural graph JSONL, target-aligned holding dataset, QA manifests.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
root = Path(os.environ.get('GRAPH_CLAD_PROJECT_ROOT', Path.cwd())).resolve()
if not (root / 'scripts').is_dir() and Path('/content/Graph-CLaD').is_dir(): root = Path('/content/Graph-CLaD')
os.chdir(root); sys.path.insert(0, str(root)) if str(root) not in sys.path else None
from scripts.research_paths import resolve_research_paths
paths = resolve_research_paths(project_root=root)
phase2_root = paths.artifact_root / 'phase2d' / 'data'
natural_root = phase2_root / 'phase2d_full_demo_v2_inputclean_stream1'
target_root = phase2_root / 'phase2d_holding_target_v2_inputclean_stream1'
split_manifest = phase2_root / 'phase2d_demo_split_manifest.json'

## 주요 설정
Graph spec은 `configs/phase2_graph_spec.json`을 기준으로 한다. Phase 2R scripted data는 extractor 진단 전용이고 training 입력에 섞지 않는다. Holding label은 contact, gripper closure, 3-frame stability, object following 기반 weak label이다.

In [ ]:
expected = {
    'graph_spec': paths.config_root / 'phase2_graph_spec.json',
    'split_manifest': split_manifest,
    'natural_root': natural_root,
    'target_root': target_root,
}
{name: {'path': str(path), 'exists': path.exists()} for name, path in expected.items()}

In [ ]:
# 전체 replay는 장시간·대용량 작업이므로 command만 구성한다.
RUN_FULL_REPLAY = False
task_hdf5 = {0: Path('SET_TASK0_HDF5'), 1: Path('SET_TASK1_HDF5'), 2: Path('SET_TASK2_HDF5')}
replay_cmd = [sys.executable, '-m', 'scripts.phase2d.build_demo_dataset',
              '--task', f"0={task_hdf5[0]}", '--task', f"1={task_hdf5[1]}", '--task', f"2={task_hdf5[2]}",
              '--split-manifest', str(split_manifest), '--bddl-root', str(paths.libero_root),
              '--output-root', str(natural_root)]
print(' '.join(map(str, replay_cmd)))
if RUN_FULL_REPLAY:
    assert all(p.exists() for p in task_hdf5.values())
    subprocess.run(replay_cmd, check=True)

In [ ]:
# 기존 release는 재생성하지 않고 manifest/shard 존재만 확인한다.
task_files = [natural_root / f'task{i}' / f'phase2d_task{i}_graph_dataset.jsonl.gz' for i in (0, 1, 2)]
target_manifest = target_root / 'phase2d_holding_target_dataset_manifest.json'
qa = {'natural_task_files': {str(p): p.exists() for p in task_files},
      'target_manifest': target_manifest.exists()}
if target_manifest.exists():
    qa['target_status'] = json.loads(target_manifest.read_text(encoding='utf-8')).get('status')
qa

## 결과 확인과 다음 phase
세 task gzip과 target manifest `status=pass`를 확인한다. 누락 시 전체 replay를 즉시 재실행하지 말고 per-demo shard/manifest부터 조사한다. 다음은 `phase_3a_dataset_and_label_qa.ipynb`다.